## Surrogate vs. Thermochemical library
**Computational speedup**

Import libraries

In [1]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

In [2]:
import utils

Define inputs

In [3]:
# "csv" header
header = ["t", "mean", "std"]
# Paths to timings
prefix = "./core/"
# Final times
tend = [1e-8, 1e-7, 1e-6]
# Timings
methods = {
  "ode": {
    "integrators": ("lsode",),
    "max_orders": np.arange(2,5).astype(str).tolist(),
    "rtol_orders": np.arange(6,9).astype(str).tolist()
  },
  "sur": {
    "models": ("meno",),
    "devices": ("cpu", "gpu")
  }
}

In [4]:
path_to_saving = os.path.abspath("./../output/speedup/")
os.makedirs(path_to_saving, exist_ok=True)

Labels

In [5]:
tend_lbl = [r"${0:s}$".format(utils.as_si(xi)) for xi in tend]
rtol_lbl = [10**(-int(rtol)) for rtol in methods["ode"]["rtol_orders"]]
rtol_lbl = [r"${0:s}$".format(utils.as_si(xi)) for xi in rtol_lbl]

Read timings

In [6]:
def read_timings(filename, header, tend=None):
  timings = pd.read_csv(filename)
  timings.columns = header
  if (tend is not None):
    timings = timings[timings['t'].isin(tend)]
  return timings

In [7]:
timingsf = {}
for method in methods.keys():
  timingsf[method] = {}
  if (method == "ode"):
    for integrator in methods[method]["integrators"]:
      timingsf[method][integrator] = {}
      for rtol in methods[method]["rtol_orders"]:
        krtol = "rtol"+rtol
        timingsf[method][integrator][krtol] = {}
        for max_order in methods[method]["max_orders"]:
          kord = "ord"+max_order
          filename = prefix + f"/{method}/data/{integrator}/timings_rtol{rtol}_ord{max_order}.csv"
          timingsf[method][integrator][krtol][kord] = filename
  elif (method == "sur"):
    for model in methods[method]["models"]:
      timingsf[method][model] = {}
      for device in methods[method]["devices"]:
        filename = prefix + f"/{method}/data/timings_{model}_{device}.csv"
        timingsf[method][model][device] = filename

In [8]:
timings = tf.nest.map_structure(lambda f: read_timings(f, header, tend), timingsf)

Compute speedup

In [9]:
speedup = {}
for model in methods["sur"]["models"]:
  speedup[model] = {}
  for device in methods["sur"]["devices"]:
    tsur = timings["sur"][model][device]
    speedup[model][device] = tf.nest.map_structure(
      lambda tode: utils.compute_speedup(tode, tsur, header), timings["ode"]
    )

In [10]:
speedup_flat = pd.json_normalize(speedup, sep="_").to_dict(orient="records")[0]

Save speedup

In [11]:
ipath = path_to_saving + "/data/"
os.makedirs(ipath, exist_ok=True)
for (ispeedup, idata) in speedup_flat.items():
  utils.save_speedup(
    df=idata,
    filename=ipath + ispeedup,
    title=f"Max order {max_order}"
  )

> LaTeX format

In [12]:
header = ["order", "rtol"] + tend_lbl
for model in methods["sur"]["models"]:
  for device in methods["sur"]["devices"]:
    for integrator in methods["ode"]["integrators"]:
      table = ""
      ispeedup = speedup[model][device][integrator]
      for order in methods["ode"]["max_orders"]:
        order_str = f"ord{order}"
        for i, rtol in enumerate(methods["ode"]["rtol_orders"]):
          rtol_str = f"rtol{rtol}"
          # Write table row
          row = ispeedup[rtol_str][order_str]["mean"].values.tolist()
          row = [order, rtol_lbl[i]] \
              + ["$"+utils.as_si(x, ndp=2)+"$" for x in row]
          # Add to table
          table += " & ".join(row) + " \\\\\n"
      filename = ipath + f"/{model}_{device}_{integrator}.txt"
      with open(filename, 'w', encoding="utf-8") as file:
        file.write(table)

> GPU/CPU ratio

In [13]:
speedup_ratio = {}
for model in methods["sur"]["models"]:
  speedup_ratio[model] = tf.nest.map_structure(
    lambda x, y: (x["mean"]/y["mean"]).to_numpy().tolist(),
    speedup[model]["gpu"],
    speedup[model]["cpu"]
  )

with open(ipath + "/speedup_ratio_gpu_cpu.json", "w") as file:
  json.dump(speedup_ratio, file, indent=2)

Plot speedup

In [14]:
ipath = path_to_saving + "/figs/"
os.makedirs(ipath, exist_ok=True)

In [15]:
for idata in speedup_flat.values():
  time = idata["t"].to_numpy().squeeze()
  break

> 1D plots

In [16]:
for order in methods["ode"]["max_orders"]:
  order = f"ord{order}"
  for model in methods["sur"]["models"]:
    for device in methods["sur"]["devices"]:
      for integrator in methods["ode"]["integrators"]:
        data = []
        ispeedup = speedup[model][device][integrator]
        for rtol in methods["ode"]["rtol_orders"]:
          krtol = f"rtol{rtol}"
          data.append(ispeedup[krtol][order]["mean"].values)
        utils.plot_timings_1d(
          x=time,
          mean=np.vstack(data),
          std=None,
          var_names=[f"RTol={rtol}" for rtol in rtol_lbl],
          labels=["$t_f$ [s]", "Speedup"],
          scales=["log", "log"],
          legend_loc="upper left",
          figname=ipath + f"/1d_{model}_{device}_{integrator}_{order}",
          save=True,
          show=False,
          extensions=("pdf",)
        )

In [17]:
for rtol in methods["ode"]["rtol_orders"]:
  krtol = f"rtol{rtol}"
  for model in methods["sur"]["models"]:
    for device in methods["sur"]["devices"]:
      for integrator in methods["ode"]["integrators"]:
        data = []
        ispeedup = speedup[model][device][integrator]
        for order in methods["ode"]["max_orders"]:
          order = f"ord{order}"
          data.append(ispeedup[krtol][order]["mean"].values)
        utils.plot_timings_1d(
          x=time,
          mean=np.vstack(data),
          std=None,
          var_names=[f"BDF{order}" for order in methods["ode"]["max_orders"]],
          labels=["$t_f$ [s]", "Speedup"],
          scales=["log", "linear"],
          legend_loc="best",
          figname=ipath + f"/1d_{model}_{device}_{integrator}_{krtol}",
          save=True,
          show=False,
          extensions=("pdf",)
        )

> 2D plot

In [18]:
for model in methods["sur"]["models"]:
  for device in methods["sur"]["devices"]:
    for integrator in methods["ode"]["integrators"]:
      ispeedup = speedup[model][device][integrator]
      for max_order in methods["ode"]["max_orders"]:
        kord = "ord"+max_order
        idata = np.vstack([df[kord]["mean"].values for df in ispeedup.values()])
        utils.plot_timings_2d(
          xlbl=tend_lbl,
          ylbl=rtol_lbl,
          z=idata,
          figsize=(7,7),
          figname=ipath + f"/2d_{model}_{device}_{integrator}_{kord}",
          save=True,
          show=False,
          extensions=("pdf",)
        )